This assignment should be done in groups of 2 or 3 (preferably 3).

This assignment consists of two parts. In the first part, your task is to solve a grid world environment using the REINFORCE algorithm. In the second part, your task is to control a lunar lander in a smooth descent to the ground. It must land in-between two flags, quickly, using minimal fuel, and without damaging the lander. You will implement the A2C algorithm that will learn a control policy for the lander.

## Imports 
Import required packages. Add additional packages if needed.

In [ ]:
import gymnasium as gym
import numpy as np

from pandas.core.config_init import is_terminal
import tqdm

import torch
import torch.distributions
import pygame


import matplotlib.pyplot as plt
from matplotlib.table import Table

## Part 1: GridWorld with REINFORCE (9 pts)
Firstly, you will implement REINFORCE in a GridWorld environment. The agent starts from the center, and the four actions left, right, up and down move the agent in the appropriate directions deterministically. The corner states are terminal and give rewards.

**In this part, you should only use `numpy` operations (i.e., no automated differentiation with PyTorch).**

### The GridWorld Environment

We already provide a basic implementation of the grid world environment which you will use,
as well as the constants `ACTION_LEFT, ACTION_RIGHT, ACTION_UP, ACTION_DOWN`.
You do not need to modify the class `GridWorld`.

In [ ]:
# DO NOT MODIFY
ACTION_LEFT = 0
ACTION_RIGHT = 1
ACTION_UP = 2
ACTION_DOWN = 3

def clip(x, a, b):
    if x < a:
        return a
    if x > b:
        return b

    return x

class GridWorld:

    def __init__(self, size=3):
        self.size = size # number of cells you can go from the center until you reach a wall
        self.state = [0,0]
        self.actions = [ACTION_LEFT, ACTION_RIGHT, ACTION_UP, ACTION_DOWN]

    def reset_world(self):
        self.state = [0,0]

    def is_terminal(self):
        return abs(self.state[0]) == self.size and abs(self.state[1]) == self.size

    def get_current_reward(self):
        if self.state == [self.size, self.size] or \
           self.state == [-self.size, -self.size]:
            return -1
        elif self.state == [-self.size, self.size] or \
             self.state == [self.size, -self.size]:
            return +1
        else:
            return 0

    def list_actions(self):
        return self.actions

    def step(self, action):
        if action not in self.actions:
            raise NotImplementedError("Action not defined")

        if self.is_terminal():
            raise NotImplementedError("Epsiode terminated! Call reset_world first.")

        if action == ACTION_LEFT:
            self.state[0] = clip(self.state[0]-1, -self.size, self.size)

        elif action == ACTION_RIGHT:
            self.state[0] = clip(self.state[0]+1, -self.size, self.size)

        elif action == ACTION_UP:
            self.state[1] = clip(self.state[1]+1, -self.size, self.size)

        elif action == ACTION_DOWN:
            self.state[1] = clip(self.state[1]-1, -self.size, self.size)

    def get_coordinates(self):
        # Assuming x goes to the right and y goes upward
        s = np.array(self.state)
        s[0] = s[0] + self.size
        s[1] = self.size - s[1]

        return s

    def get_state(self):
        return self.get_coordinates()[0] + \
               self.get_coordinates()[1] * (self.size * 2 + 1)

    def render_state(self):
        N = self.size
        x = self.state[0]
        y = self.state[1]

        fig, ax = plt.subplots(figsize=(5,5))
        ax.set_axis_off()

        tb = Table(ax, bbox=[0,0, 1, 1])

        row_labels = np.arange(2 * N + 1)
        col_labels = np.arange(2 * N + 1)
        width, height = .5, .5

        values = np.zeros((N*2+1, N*2+1))
        values[0,0] = values[-1, -1] = 1
        values[-1,0] = values[0, -1] = -1

        cell_colors = ('white', 'whitesmoke')
        font_colors = ('black', 'white')

        for (row_idx, col_idx), cell_val in np.ndenumerate(values):
            idx = (col_idx + row_idx) % 2
            if cell_val !=  0.:
                text = "%.1f" % cell_val
                if cell_val > 0:
                    c = "lightblue"
                else:
                    c = "lightcoral"
            elif row_idx == N and col_idx == N:
                text = "start"
                c = "antiquewhite"
            else:
                text = ""
                c = cell_colors[idx]

            tb.add_cell(row_idx, col_idx, width, height,
                        text=text,
                        loc='center',
                        facecolor=c)

        for (row_idx, col_idx), cell_val in np.ndenumerate(values):
            idx = (col_idx + row_idx) % 2
            tb._cells[(row_idx, col_idx)]._text.set_color(font_colors[idx])

        pos = plt.Circle((0.5 + 1/(2 * N + 1) * x, 0.5 + 1/(2 * N + 1) * y + .04), 0.01, color='black')
        ax.add_patch(pos)
        ax.add_table(tb)

In [ ]:
# You can plot the environment with render_state()
GridWorld().render_state()

In [ ]:
# The state after the agent takes the action Right
world = GridWorld()
world.reset_world()
world.step(ACTION_RIGHT)
world.render_state()

In [ ]:
# cells/states are numbered from 0 to 48, left to right, top to bottom
world.get_state()

In [ ]:
# for convenience, cells are given coordinates left to right, top to bottom
world.get_coordinates()

### Direct Parameterization

In this part, we will parametrise the policy according to a direct (exponential) softmax distribution:
$$
\pi_\theta(a|s) = \frac{\exp{\theta^\top \phi_{s,a}}}{\sum_{a'} {\exp{\theta^\top \phi_{s,a'}}}}
$$

We will assume that the vectors $\phi_{s,a} \in \mathbb{R}^{|\mathcal{S}||\mathcal{A}|}$ are encoded as standard unit basis vectors (i.e., one-hot encoding).
First, we will create an appropriate encoding function for the states in GridWorld.

For consistency, we assume the actions are mapped to integers with `ACTION_LEFT=0, ACTION_RIGHT=1, ACTION_UP=2, ACTION_DOWN=3`, and the cell states are given values from 0 to 48, starting from left to right, from the top row to the bottom row. 
Note that `GridWorld.get_state()` implements the correct order on grid cells.
$\phi_{s,a}$ is assumed to be the unit vector with only the entry at position $4\times s + a$ equal to 1.




**Task 1.1 (0.5 pt):** Create a function `get_features_onehot` that encodes each $\phi_{s,a}$ with a one hot encoded vector in $\mathbb{R}^{|\mathcal{S}||\mathcal{A}|} = \mathbb{R}^{196}$ as a numpy vector.

In [ ]:
def get_features_onehot(state:int, action:int):
    """Given the tuple (x,y) indicating the coordinates on the grid world, return one hot encoded features

    Assume actions are numbered from 0 to 3, and states are numbered from 0 to 48
    as explained above.
    """

    # TODO
    phi = np.zeros(196)
    phi[4*state + action] = 1


    return phi

**Task 1.2 (0.5 pt):** Implement an initialisation for the parameters $\theta \in \mathbb{R}^{196}$ as a numpy vector that you think is appropriate for efficient exploration.

In [ ]:
def get_initial_parameters():
    """Returns vector of size 196 of initial parameters theta"""
    theta = np.zeros(196)
    return theta

**Task 1.3 (2 pts):** 
First, calculate the partial derivative $\frac{\partial \pi_\theta(a|s)}{\partial \theta_{s',a'}}$. Simplify the expression as much as possible, ideally until it only contains policy probabilities ($\pi(\cdot|\cdot)$) and indicator functions ($\mathbb{1}[\cdot = \cdot]$). In other words, no sums or exponents should remain. Show your computation steps! 

Then, using your expression, fill in `direct_parameterisation_derivative`, that will return the vector $\frac{\partial \pi_\theta(a|s)}{\partial \theta}$ given a value of $\theta$ and state-action pair $s,a$.

**Hint:** The function `direct_parameterisation_derivative` should return a vector of size $4 |S| = 4 \times 7^2=196$, since you have 4 actions and $7^2$ states.

**Hint:** Only 4 entries of $\frac{\partial \pi_\theta(a|s)}{\partial \theta} \in \mathbb{R}^{196}$ should be nonzero for fixed $a,s$.

TODO
We look for $\dfrac{\partial \pi_\theta(a|s)}{\partial \theta_{s',a'}}$.

*1.* $\phi_{s,a}$ is one-hot with a 1 at position $4s+a$ and 0 elsewhere, so $\theta^\top \phi_{s,a} = \theta_{s,a}$, the value of $\theta$ at position $4s+a$.

*2.* Policy becomes:

$$\pi_\theta(a|s) = \frac{e^{\theta_{s,a}}}{\sum_b e^{\theta_{s,b}}}$$

With $f = e^{\theta_{s,a}}$ and $g = \sum_b e^{\theta_{s,b}}$, we get:

$$\frac{\partial \pi_\theta(a|s)}{\partial \theta_{s',a'}} = \frac{f'g - fg'}{g^2}$$

*3.* Three cases: $s' \neq s$; $s' = s, a' = a$; $s' = s, a' \neq a$.

$s' \neq s$:

$$\frac{\partial \pi_\theta(a|s)}{\partial \theta_{s',a'}} = \frac{\frac{\partial e^{\theta_{s,a}}}{\partial \theta_{s',a'}} \sum_b e^{\theta_{s,b}} - e^{\theta_{s,a}} \frac{\partial \sum_b e^{\theta_{s,b}}}{\partial \theta_{s',a'}}}{\left(\sum_b e^{\theta_{s,b}}\right)^2} = \frac{0 \cdot g - f \cdot 0}{g^2} = 0$$

$s' = s,\ a' = a$:

$$\begin{aligned}
\frac{\partial \pi_\theta(a|s)}{\partial \theta_{s,a}}
&= \frac{\frac{\partial e^{\theta_{s,a}}}{\partial \theta_{s,a}} \sum_b e^{\theta_{s,b}} - e^{\theta_{s,a}} \frac{\partial \sum_b e^{\theta_{s,b}}}{\partial \theta_{s,a}}}{\left(\sum_b e^{\theta_{s,b}}\right)^2}
= \frac{e^{\theta_{s,a}} \sum_b e^{\theta_{s,b}} - e^{\theta_{s,a}} e^{\theta_{s,a}}}{\left(\sum_b e^{\theta_{s,b}}\right)^2}
&= \frac{e^{\theta_{s,a}}}{\sum_b e^{\theta_{s,b}}} - \frac{e^{\theta_{s,a}}}{\sum_b e^{\theta_{s,b}}} \cdot \frac{e^{\theta_{s,a}}}{\sum_b e^{\theta_{s,b}}}
= \pi_\theta(a|s) - \pi_\theta(a|s)\,\pi_\theta(a|s)
= \pi_\theta(a|s)\big(1 - \pi_\theta(a|s)\big)
\end{aligned}$$

$s' = s,\ a' \neq a$:

$$\begin{aligned}
\frac{\partial \pi_\theta(a|s)}{\partial \theta_{s,a'}}
&= \frac{\frac{\partial e^{\theta_{s,a}}}{\partial \theta_{s,a'}} \sum_b e^{\theta_{s,b}} - e^{\theta_{s,a}} \frac{\partial \sum_b e^{\theta_{s,b}}}{\partial \theta_{s,a'}}}{\left(\sum_b e^{\theta_{s,b}}\right)^2}
= \frac{0 \cdot \sum_b e^{\theta_{s,b}} - e^{\theta_{s,a}} e^{\theta_{s,a'}}}{\left(\sum_b e^{\theta_{s,b}}\right)^2}
&= -\frac{e^{\theta_{s,a}}}{\sum_b e^{\theta_{s,b}}} \cdot \frac{e^{\theta_{s,a'}}}{\sum_b e^{\theta_{s,b}}}
= \pi_\theta(a|s)\big(0 - \pi_\theta(a'|s)\big)
\end{aligned}$$

Combining three cases:

$$\frac{\partial \pi_\theta(a|s)}{\partial \theta_{s',a'}} =
\begin{cases}
0 & \text{if } s' \neq s \\
1 \cdot \pi_\theta(a|s)\big(1 - \pi_\theta(a|s)\big) & \text{if } s' = s,\ a' = a \\
1 \cdot \pi_\theta(a|s)\big(0 - \pi_\theta(a'|s)\big) & \text{if } s' = s,\ a' \neq a
\end{cases}$$

which can be written as

$$\frac{\partial \pi_\theta(a|s)}{\partial \theta_{s',a'}} = \mathbf{1}[s' = s]\,\pi_\theta(a|s)\big(\mathbf{1}[a' = a] - \pi_\theta(a'|s)\big)$$


In [ ]:
def softmax(weights,action_vectors,action_done):

    prob = np.exp(action_vectors[action_done]@weights)/(np.sum(np.exp(action_vectors@weights)))
    return prob

def direct_parameterisation_gradient(action: int, state: int, theta: np.ndarray):
    """Returns the partial derivatives of the policy with respect to theta

    Assume actions are numbered from 0 to 3, and states are numbered from 0 to 48
    as explained above.
    """
    gradient_vector = np.zeros(196)
    action_vectors = np.array([get_features_onehot(state,a) for a in range(4)])

    for act in range(4):
        idx = 4*state + act
        if act == action:
            gradient_vector[idx] = softmax(theta,action_vectors,act)* (1 - softmax(theta,action_vectors,action))*action_vectors[act,idx]
        else:
            gradient_vector[idx] = -softmax(theta,action_vectors,act)*  softmax(theta,action_vectors,action)*action_vectors[act,idx]
    
    return gradient_vector

    
def direct_parameterisation(action,state, theta):
    action_vectors = np.array([get_features_onehot(state,a) for a in range(4)])
    prob = softmax(theta,action_vectors,action)
    return prob

    # TODO

The code below helps you check if your derivative implementation is correct.
In case your implementation works, the code should print success.
You should not modify the cell below.

In [ ]:
## checks the code above

def check():
    torch.manual_seed(0)
    success = True
    N = 10
    err = 0.

    for i in range(N):
        theta = torch.rand((4*49))
        theta.requires_grad = True
        theta_shaped = theta.reshape(49,4) # (n_states, n_actions)

        for state in range(49):
            for action in range(4):
                # Although we can do this computation for all states at once, autograd requires us to do it every time since we reset the gradients
                theta_state = theta_shaped[state]  # (n_actions,)
                theta_state_exp = torch.exp(theta_state) # (n_actions,)
                action_probs = theta_state_exp / theta_state_exp.sum(-1, keepdim=True)

                action_probs[action].backward()

                answer = direct_parameterisation_gradient(action, state, theta.detach().cpu().numpy())

                err += np.abs(answer - theta.grad.cpu().numpy()).sum()

                if not np.allclose(answer, theta.grad.cpu().numpy()):
                    success = False

                theta.grad.zero_()

    if success:
        print(f"Success: errors within tolerance. Mean abs. error: {err / (4 * 49 * N)}")
    else:
        print(f"Fail: errors not within tolerance. Mean abs. error: {err / (4 * 49 * N)}")

    del theta, theta_shaped, action_probs, answer

check()

### REINFORCE Implementation

**Task 1.4 (0.5 pt):** In the lectures, you have been introduced to the REINFORCE algorithm. Explain what the regular REINFORCE objective aims to optimise and how it is achieved only using sample and no prior information on the MDP dynamics. 
*Write in the Markdown cell below (use LaTeX-math mode for equations, etc.).*

The REINFORCE-algorithm aims to optimize the state-value function for the starting state which is dependent on the policy. Therefore we try to improve our policy to maximize this function. In our case we have a softmax function of a linear layer with weights which are multiplied by the one hot encoding vector. By changing the weight to improve the state-value function we can achieve this. This is done with the following algorithm:
$\theta <- \theta + \alpha \gamma^t G \nabla ln(\pi(a|s)) $
This works on the idea by taking the gradient of the state-value function with respect to the weights we can update these weights in the direction where the state value function increases the most at each step. This the update uses the policy-gradient theorem which is a analytic expression for the derivative of the state-value that contains no derivative of the state distribution. By sampling over different epsiodes, and updating after each episode this creates the same effect as calculation the state-distribution would. This is because the state distribution both depends on the policy and the environment, and therefore sampling the value of $\gamma^t G \nabla ln(\pi(a|s))$ is a unbiased estimate of the gradient of the state-value. Some episodes might be noisy but over many episods it converges to the true value.

**Task 1.5 (3 pts):** Using your functions implemented in Task 1.1, 1.2 and 1.3, fill in the `ReinforceLearner` class. You can use the already implemented methods of the GridWorld class: 
- `reset_world`: reset the world to starting state
- `is_terminal`: is the current state terminal
- `get_current_reward`: reward of the current state
- `step`: given an action among `ACTION_LEFT, ACTION_RIGHT, ACTION_UP, ACTION_DOWN`, takes a step in the simulation
- `get_state`: get current state, as an integer from 0 to 48
- `get_coordinates`: get current coordinates in table as a tuple (x,y)
- `render_state`: render the current state of the grid world, useful for debugging

**Hint:** You do not need to implement an exploration strategy with the appropriate intialisation, so only vanilla policy gradient steps should be sufficient for convergence with an appropriate learning rate.

In [ ]:
class ReinforceLearner:

    def __init__(self, learning_rate, gamma, seed = None):
        """Intialisation code"""

        self.world = GridWorld()
        self.learning_rate = learning_rate
        self.gamma = gamma
        self.theta = get_initial_parameters()
        self.rng = np.random.default_rng(seed)

        ## TODO: you may use other variables or inputs as your hyperparameters
        # TODO

    def get_action_prob(self, state: int):
        """Given the state, return the probability of taking each of the 4 actions"""

        # TODO
        action_vectors = np.array([get_features_onehot(state,a) for a in range(4)])
        probability= [ softmax(self.theta,action_vectors,a) for a in range(4)]
        return probability

    def run_episode(self, visualize_steps = False):
        self.world.reset_world()
        actions = []
        states = []
        rewards = []
        more_steps= True
        while more_steps:
            if visualize_steps:
                self.world.render_state()
                print(" ")

            current_state = self.world.get_state()
            action_probs = self.get_action_prob(current_state)
            action = self.rng.choice(self.world.list_actions(),p=action_probs )
            self.world.step(action)

            actions.append(action) 
            states.append(current_state)
            rewards.append(self.world.get_current_reward())
            
            if self.world.is_terminal():
                more_steps = False


        if visualize_steps:
            self.world.render_state()

        actions = np.array(actions)
        states = np.array(states)
        rewards = np.array(rewards)

        assert actions.shape == states.shape and states.shape ==rewards.shape
        return  actions,states, rewards

    def train(self, n_updates:int, n_trajectories:int):
        """Run the training procedure for `n_updates` steps

        n_trajectories: how many trajectories should be sampled each REINFORCE step
        n_updates: how many parameters updates to make (each parameter update would sample n_trajectories)"""

        rewards = []
        for episode in range(n_updates):
            rewards.append(self.reinforce_step(n_trajectories))

        return rewards

    def reinforce_step(self, n_trajectories: int):
        """Run a single gradient step for REINFORCE (without baseline) using n_trajectories many trajectories for estimation.

        Also return the average reward observed"""

        # TODO
        action_trajectorys = []
        state_trajectorys = []
        reward_trajectorys = []
        reward_per_round = []
        for episode in range(n_trajectories):
            actions, states, rewards = self.run_episode()

            action_trajectorys.append(actions)
            state_trajectorys.append(states)
            reward_trajectorys.append(rewards)
            reward_per_round.append(np.sum(rewards))

        delta_theta = np.zeros_like(self.theta)

        for episode in range(n_trajectories):
            delta_theta+= self.get_delta_theta(action_trajectorys[episode],state_trajectorys[episode],reward_trajectorys[episode])
        self.theta = self.theta + delta_theta 
        return np.mean(reward_per_round)

    def get_delta_theta(self, actions,states,rewards):
        delta_theta = np.zeros(196)
        trajectory_length = len(rewards)
        g = 0
        for i in range(trajectory_length):
            idx = trajectory_length - i -1
            g = g* self.gamma + rewards[idx]
            delta_theta += self.learning_rate * self.gamma**idx * g* direct_parameterisation_gradient(
                    actions[idx],states[idx],self.theta)/direct_parameterisation(
                    actions[idx],states[idx],self.theta)
            
        return delta_theta

    def evaluate(self, render: bool = True):
        """Evaluate current algorithm for a single episode and return.

        You should
        (a) Print the total (undiscounted) reward from the episode.
        (b) Visualise the whole state history. (Hint: You can use multiple calls to GridWorld.render_state())"""
        actions, states, rewards = self.run_episode(visualize_steps= render)
        print("Total reward: ",np.sum(rewards))

       # TODO

### Evaluation

**Task 1.6 (1 pt):** Run the above code until convergence, and evaluate your strategy at the end.
Plot the average reward observed per epoch during training.
You made need to tune your hyper-parameters.

In [ ]:
# Remember to evaluate it at the end. Show your evalutation (including visualization of state history)!
# TODO
seed = 42
steps = 200
batch = 10
lr = 0.1
gamma = 0.99
agent = ReinforceLearner(lr,gamma,seed = seed)
mean_rewards = agent.train(steps,batch)
agent.evaluate()

In [ ]:
## Plot your rewards for your run of 1.5 here
# TODO
x = np.arange(0,steps,1)
fig,ax = plt.subplots(figsize= (5,5))
ax.plot(x,mean_rewards)
ax.set_xlabel("Steps")
ax.set_ylabel("Mean reward")
ax.set_title("Reward over each batch")

### Additional Questions

**Task 1.7 (1 pt):** The code above for REINFORCE does not implement an explicit exploration strategy.
Explain how the exploration-exploitation trade-off was circumvented by your choice of parameters intialisation in two sentences, and how this approach could fail for some initialisations.
*Write in the Markdown cell below (use LaTeX-math mode for equations, etc.).*


If you initalize the weights to be uniform, the agent will in the beginning before updating anything, every action totaly random because of the softmax function. If the the initalizations are higher in some areas, and these will get prioritized and therefore the agent will explore these more than where the values are lower.

**Task 1.8 (0.5 pt):** In a few sentences explain the shortcomings of the direct parametrisation (one parameter for each state-action pair).
*Write in the Markdown cell below (use LaTeX-math mode for equations, etc.).*

One big drawback is that all the states are looked at totaly independent as they are not connected in any way, which means that generalization can't happen meaning that what the agent learns in one state isn't being transfered to the other. Another drawback is the dimensionality problem of this. It needs 4 different weights for every state so if one would scale this to a larger environment the weights would increase with $4*n^2$ were $n$ is the number of cells per side.


##  Part 2: CartPole with A2C (9 pts)

### The CartPole Environment

The [CartPole](https://gymnasium.farama.org/environments/classic_control/cart_pole/) environment is a classical problem where a pole is attached by an un-actuated joint to a cart, which moves along a frictionless track. The pendulum is placed upright on the cart and the goal is to balance the pole by applying forces in the left and right direction on the cart.

Since the focus of this task is the implementation of reinforcement
learning algorithms, it is not necessary to have a detailed understanding of the mechanics of the
Cart Pole environment beyond the observation and action space sizes.

### Initialize the environment
Initialize the Cart Pole environment in the Gymnasium library to start learning a policy for it. You can read more about the Gymnasium library [here](https://gymnasium.farama.org/).

In [ ]:
env = gym.make("CartPole-v1")

### PyTorch Intro

We also introduce basic functionality for PyTorch.
PyTorch is a widely used automatic differentiation library, which is very useful for training neural networks.
This part is only for guidance and does not include questions.
While we do introduce the basics of PyTorch, you are encouraged to explore more about the library following the official [tutorials](https://pytorch.org/tutorials/beginner/basics/intro.html) in case you haven't used it before.

PyTorch is built on top of the class `torch.Tensor`, which implements many operations on vectors, matrices and higher dimensional tensors.
The functionality of operations on torch tensors closely mirrors the operations from the `numpy` package.

In [ ]:
torch.zeros(5)

In [ ]:
a_matrix = [[1., 2.],[3., 4.]]
torch.tensor(a_matrix)

In [ ]:
torch.matmul(torch.tensor(a_matrix),
             torch.tensor([4., 5.]))

What makes torch special is that it implements automated differentiation by tracking the operation history on tensors.
Namely, if a `torch.Tensor` has the propery `.requires_grad` set to `True`, torch will automatically track operations made on that tensor.
A final `.backward()` call to a computation result will compute partial derivatives of the variable with respect to all tensors in the tracked computational graph.
This is particularly useful for tracking derivatives with respect to a loss function, to enable simple gradient descent.

In [ ]:
# example from https://pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html

x = torch.ones(5)  # input tensor
y = torch.zeros(3)  # expected output
w = torch.randn(5, 3, requires_grad=True)
b = torch.randn(3, requires_grad=True)
z = torch.matmul(x, w)+b
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)
print(loss)

In [ ]:
# differentiate with respect to loss
loss.backward()
# the computed partial derivatives of w
w.grad

To prevent tracking history, you can also wrap a piece of code in`with torch.no_grad():`. 
This can be particularly helpful when evaluating a model on new data, when training (and hence differentiation) is not required.

For more advanced modelling (as will be required by this exercise), it is usually useful to implement classes as children of `nn.Module`.
These implement the function `forward`, which returns the output of the forward pass of a model given inputs.
Modules have `nn.Parameter`, which will be the trainable parameters of the models.
Most useful sub-modules are already provided by the torch package: see for instance `nn.Linear`.

Finally, torch implements many standard optimizers: for instance `torch.optim.SGD`, `torch.optim.Adam`.
These are typically used to automatically take care of training once partial derivatives are computed.

### Advantage Actor-Critic (A2C)

A more stable alternative to vanilla policy gradients via REINFORCE is the so-called A2C algorithm, which is an actor-critic method.
First, you should familiarise yourself with the following paper [paper](https://arxiv.org/pdf/1602.01783.pdf) which introduces A3C (Asynchronous Advantage Actor Critic). A2C was later introduced as a simplification of A3C, see the following resources: [OpenAI Baselines: ACKTR & A2C](https://web.archive.org/web/20230327182809/https://openai.com/research/openai-baselines-acktr-a2c) and [Lil'Log: Policy Gradient Algorithms](https://lilianweng.github.io/posts/2018-04-08-policy-gradient/#a2c)

**Task 2.1 (1.5 pts):** Explain in a few sentences each, 

1. how the A2C algorithm addresses the shortcomings of vanilla policy gradient methods,
2. the difference between A2C and A3C,
3. why the A3C algorithm could be expected to perform even better than A2C.

TODO

1. The A2C uses multiple diffrent training instances to create the the updates of the weightswhich creates a more diversive training data than just one agent training. It's a much more efficient way of creating the same effects as experience replay. It also implements n-step updates which means that you can have online training, and full Monte Carlo returns can be very noisy, and therefore the learning is much more stabel. It also because it's a actor-critic model, creates an estimate for both the state-value function and the policy with separate models, where the critice acts like a benchmark that varies over all the states. Because of the critic that is trained, the model has a baseline were each reward can be compared to the baseline, and because of this, we get negative steps for the gradient, for actions that perform worse than baseline.
2. The difference between A2C and A3C is the word Asynchronous that is missing for the A2C. What this means is that the A2C runs all the agents n-steps and waits until everybody is finished and then it updates the weights with all the accumulated gradients, but the A3C does not wait for every agent to be done but instead each agent updates the global weights as soon as they are done and then starts training again which means that the different agent can be opperating on the environment with different weights at the same time.
3. This could be the case because for A3C different agents can have different weights at the same time, the behaviour of the different agents will be more diverse and therefore we could think that the model experience more exploration for the same price. But as written in some of the reports above, the they have only experienced evidence of A2C being superior to A3C.

### Neural Network Parameterization

We will use neural networks to parameterize both the value function and the policy.
The `ActorNet` defines the policy $\pi_\theta(a|s)$. Given state $s$ as input, it will output a distribution over the action space. 
The `CriticNet` defines the approximation to the value function, which could be denoted $\hat{V}_w(s)$. 

You can play with different network architectures and the [activation](https://pytorch.org/docs/stable/nn.html) functions.
Note that the `ActorNet` should produce a probability distribution over the action space.

**Task 2.2 (0.5 pts):** Design your own neural networks for `ActorNet` and `CriticNet` in the code block below.

In [ ]:
class ActorNet(torch.nn.Module):
    def __init__(self, num_state: int, num_action: int):
        super(ActorNet, self).__init__()

        # see torch.nn.Sequential, for a hint
        # TODO
        self.layers = torch.nn.Sequential(torch.nn.Linear(num_state,64),torch.nn.ReLU(),torch.nn.Linear(64,16),torch.nn.ReLU(),torch.nn.Linear(16,num_action),torch.nn.Softmax(dim = -1))



    def forward(self, state):
        output = self.layers(state)
        return output


class CriticNet(torch.nn.Module):
    def __init__(self, num_state: int):
        super(CriticNet, self).__init__()
        self.layers = torch.nn.Sequential(torch.nn.Linear(num_state,32),torch.nn.ReLU(),torch.nn.Linear(32,8),torch.nn.ReLU(),torch.nn.Linear(8,1),torch.nn.ReLU())


        # see torch.nn.Sequential, for a hint
        # TODO

    # return a single value
    def forward(self, state):
        output = self.layers(state)
        return output
        # TODO

### Learning Actor and Critic
We train both the actor and critic nets by interacting with the environment.
You should fill in the marked places below to implement the A2C training process.

**Task 2.3 (5 pts):** 
Fill in the code below to implement A2C in the Cart Pole environment.
You should play with your hyper-parameters as well, to enable efficient learning.
You may play with different learning rates and optimizers.
For different optimizers in PyTorch, see [here](https://pytorch.org/docs/stable/optim.html).
Note that the agent should solve the problem in under 1000 episodes.

In [ ]:
device = torch.device('cpu' if torch.cuda.is_available() else 'cpu')
torch.set_default_device(device)

print(device)

In [ ]:
from gymnasium import Env


class A2C():

    def __init__(self, env: Env, tau = 0.0) -> None:
        """Initialize the A2C agent in the environment env. The argument tau is used later in task 2.5. 

        Args:
            env: The environment the agent will use. 
            tau: The entropy coefficient for task 2.5.
        """
        self.obs_dim = env.observation_space.shape[0]
        self.action_dim = env.action_space.n
        self.env = env
        self.tau= tau
        self.actor = ActorNet(self.obs_dim,self.action_dim)
        self.critic = CriticNet(self.obs_dim)
        self.optimizer = torch.optim.Adam([{"params":self.actor.parameters(), "lr":1e-3},{"params":self.critic.parameters(),"lr": 7e-3}], weight_decay=1e-3)
        self.n_step = 1000
        self.gamma  = 0.99



    def train(self, n_episodes, max_episode_len):
        """Train the A2C agent in the given environment.

        Return the total reward for each episode.

        Args:
            n_episodes: number of episodes to train
            max_episode_len: maximum length of each episode
        """
        rewards_each_episode = []
        for episode in range(n_episodes):
            rewards_each_episode.append(np.sum(self.run_episode(max_episode_len)))

        return rewards_each_episode
            
        
    def run_episode(self,max_length: int):
        rewards = []
        current_state, _ = self.env.reset()
        index = 0

        while index < max_length:
            steps_left = max_length - index
            new_rewards, new_state, end_episode= self.run_n_steps(current_state,steps_left)
            rewards += new_rewards

            current_state = new_state
            index+= len(rewards)

            if end_episode:
                break

        return rewards

    def run_n_steps(self,first_state,steps_left):
        states = []
        actions = []
        rewards = []
        categoricals = []
        n = np.min([steps_left,self.n_step])
        end_episode = False
        current_state = first_state


        for i in range(n):
            probs = self.actor(torch.tensor(current_state).to(device)).squeeze()
            m = torch.distributions.Categorical(probs)
            action = m.sample().cpu().numpy()
            observations,reward, terminated, truncated, _ = self.env.step(action)
            states.append(current_state) 
            actions.append(action.item())
            rewards.append(reward)
            categoricals.append(m)

            current_state = observations


            if terminated:
                large_r= 0
                end_episode = True
                break
            elif i == n-1:
                large_r = self.critic(torch.tensor(current_state).to(device)).detach()


        states,actions,rewards = torch.tensor(np.array(states)).to(device), torch.tensor(np.array(actions)).to(device) ,torch.tensor(np.array(rewards)).to(device)
        
        self.update_step(states,actions,rewards, categoricals,large_r)

        return rewards.cpu(), current_state,end_episode



    def update_step(self, states: torch.Tensor, actions: torch.Tensor, rewards: torch.Tensor, categoricals,large_r: int,) -> None:
        """Perform a single update step for both actor and critic networks. Loss is averaged over the provided trajectory.

        Note that we do not require the actions themselves, since log probs can be computed from action indices and the policy distribution.

        Args:
            states: tensor of shape (T, obs_dim) containing the states visited at each time (S0, S1, ..., ST-1)
            actions: tensor of shape (T,) containing the *index* of action taken at each time step (A0, A1, ..., AT-1)
            rewards: tensor of shape (T,) containing the rewards obtained at each time step (R1, R2, ..., RT)
        """

        assert len(states) == len(actions) == len(rewards)
        T = len(states)

        assert states.shape == (T, self.obs_dim)
        assert actions.shape == (T,)
        assert rewards.shape == (T,)

        # TODO
        self.optimizer.zero_grad()
        for i in range(T):
            index = T -i -1
            critic_output = self.critic(states[index])
            large_r = rewards[index] + self.gamma * large_r
            h =  categoricals[index].entropy()
            loss_actor = (-categoricals[index].log_prob(actions[index])*(large_r - critic_output.detach()) -self.tau*h)/T
            loss_critic = (large_r -critic_output)**2/T
            loss_actor.backward()
            loss_critic.backward()
        self.optimizer.step()

We use a simple criteria to judge whether you solve the problem.
We will compute an average of the total reward of the previous 100 episodes at each iteration.
If there exists an iteration with average total reward larger than a threshold, as shown below, we will give you full grades.
That is, the problem is solved when the average reward is greater than or equal to 475 over 100 consecutive trials.
The agent must solve the environment using at most 1000 episodes.

Note that `CartPole-v1` has a termination condition of 500 timesteps. It’s done so that one episode doesn’t take forever.
So we say that if a policy can balance a pole for 500 time steps (and achieve 500 reward) it’s probably good enough.

In [ ]:
# Run your implementation here.
def was_run_successful(episodic_rewards):
    # this function should print success
    success_flag = False
    success_iter = None
    buffer_len = 100
    buffer_sum = sum(episodic_rewards[:buffer_len])
    avg_reward = [buffer_sum / buffer_len]
    max_num_episodes = 1000

    for num_iter in range(buffer_len, n_episodes):
        buffer_sum += episodic_rewards[num_iter]
        buffer_sum -= episodic_rewards[num_iter - buffer_len]
        buffer_avg = buffer_sum / buffer_len
        avg_reward.append(buffer_avg)

        if buffer_avg >= env.spec.reward_threshold and not success_flag:
            if num_iter +1 <= max_num_episodes:
                print(f'The agent successfully solved the problem in {num_iter+1} iterations!')
            else:
                print(f'Too slow! The agent solved the problem in {num_iter + 1} iterations, but must solve it under {max_num_episodes}.')    
            success_flag = True
            success_iter = num_iter+1

    if not success_flag:
        print('Unfortunately the agent is not smart enough.')
    # plot the average reward curve through the training process
    plt.plot(range(buffer_len-1, len(episodic_rewards)), avg_reward)
    plt.axhline(env.spec.reward_threshold, color = 'red', linestyle = "--", zorder = -3, lw = 1)
    if success_iter:
        plt.axvline(success_iter, color = 'red', linestyle = "--", zorder = -3, lw = 1)
    plt.show()
n_episodes = 1000
max_episode_len = 500
a2c = A2C(env)
episodic_rewards = a2c.train(n_episodes, max_episode_len)
was_run_successful(episodic_rewards)

**Useful hints:**

1. See [here](https://pytorch.org/docs/stable/distributions.html) for distributions in PyTorch. You may find the function `log_prob()` useful. You can easily compute $\log\pi(a|s)$ and its gradient using the function.

2. To optimize a model in PyTorch, you first define the loss function, then call the corresponding optimizer and perform backpropagation. For example, to optimize over the value function approximation, first let $\text{loss}=\|V_w-V^\pi\|^2$ and then call `loss.backward()` to compute the gradient.

3. You may find the function `detach()` defined in PyTorch useful if you need to call `backward()` multiple times on the same variables without storing the computational graph every time.

4. Maximizing $f(x)$ is equivalent to minimizing $-f(x)$. The default optimizers in PyTorch perform minimization.

5. You may find the classes `torch.nn.Softmax`, `torch.nn.ReLU`, `torch.nn.Tanh` useful when implementing a neural network architecture.

6. You may experiment with epsilon greedy and possibly different decay rates for the epsilon parameter.

### Analyze your results
You can run the cell below to visualize your policy and help you understand what your policy is doing.

In [ ]:
from IPython.display import clear_output
def visualize_policy(a2c: A2C):
    #env_wrap = gym.wrappers.RecordVideo(env, './results')
    env_render = gym.make("CartPole-v1", render_mode="rgb_array")
    for num_episode in range(1):
        s,_ = env_render.reset()
        tot_reward = []
        for t in range(max_episode_len):
            #env.render()

            state = torch.tensor(s, dtype=torch.float).view(1, -1)
            prob = a2c.actor(state)
            policy = torch.distributions.categorical.Categorical(prob)
            action = policy.sample()
            s, r, is_terminal, _,_ = env_render.step(action.item())
            tot_reward.append(r)
            if is_terminal:
                print("Episode {} terminated after {} steps with reward {}. ".format(num_episode, t+1, sum(tot_reward)))
                break
            clear_output(wait=True)
            plt.imshow(env_render.render())
            plt.show()

    env.close()
visualize_policy(a2c)

### Entropy regularized A2C

The exploration-exploition trade-off is also a significant problem to be solved in the context of policy gradients.
There have been numerous approaches in literature aimed towards maximizing exploration in a principled and theoretically meaningful manner.
One such example is the entropy regularised policy gradient objective:
$$
J_\tau(\theta) = J(\theta) + \tau \mathbb{E}_{s \sim d^\pi_s}\left[-\sum_a \pi_\theta(a|s) \log \pi_\theta(a|s)\right],
$$
where $J(\theta)$ is the vanilla policy gradient objective and $\tau$ is a hyperparameter controlling regularisation.

**Task 2.4 (0.5 pt):** In a sentence or two, explain why entropy regularization is relevant for encouraging exploration.

*Write in the Markdown cell below (use LaTeX-math mode for equations, etc.).*

This is because the model often converges fast to a policy that is non-optimal but easy to find, and therefore to combat this, we introduce this extra reward to favor more entropy, and therefore we only get conversion if the policy is good enough to be better than this extra reward.


**Task 2.5 (1.5 pt):** Modify the training code from above to demonstrate A2C with entropy regularization. Implement a new loss function that returns your modified loss with entropy regularization. Use entropy regularisation on the CartPole task. Visualize your results in **the cell below** and show that you have solved the problem. As before, the agent must achieve an average reward of 475 over 100 consecutive episodes. The agent can train on at most 1000 episodes.

Also feel free to explore different values of $\tau$ and it's effect on the policy optimization process.

In [ ]:
# Run your implementation here.
n_episodes = 1000
max_episode_len = 500
tau = 0.05 # TODO: Set to non-zero value.
a2c_ent = A2C(env, tau)
episodic_rewards_ent = a2c_ent.train(n_episodes, max_episode_len)
was_run_successful(episodic_rewards_ent)